In [1]:
import os
# Debugging: Print the resolved project root path
print("Resolved project root:", os.path.abspath(os.path.join(os.getcwd(), '../utils')))

Resolved project root: c:\ARKO\Projects\GitHub\MIMIC-III\Federated_Foundation_Models\src\Safe Offline Reinforcement Learning with Hierarchical and Multimodal Architectures for Personalized Sepsis Care\src\utils


In [2]:
# Debugging: Print the current working directory
print("Current working directory:", os.getcwd())

Current working directory: c:\ARKO\Projects\GitHub\MIMIC-III\Federated_Foundation_Models\src\Safe Offline Reinforcement Learning with Hierarchical and Multimodal Architectures for Personalized Sepsis Care\src\notebooks


In [3]:
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../utils')))
print("Updated sys.path:", sys.path)

Updated sys.path: ['c:\\ARKO\\Projects\\GitHub\\MIMIC-III\\Federated_Foundation_Models\\src\\Safe Offline Reinforcement Learning with Hierarchical and Multimodal Architectures for Personalized Sepsis Care\\src\\utils', 'c:\\ARKO\\Projects\\GitHub\\MIMIC-III\\Federated_Foundation_Models\\src\\Safe Offline Reinforcement Learning with Hierarchical and Multimodal Architectures for Personalized Sepsis Care\\src\\notebooks', 'c:\\ProgramData\\anaconda3\\python312.zip', 'c:\\ProgramData\\anaconda3\\DLLs', 'c:\\ProgramData\\anaconda3\\Lib', 'c:\\ProgramData\\anaconda3', '', 'C:\\Users\\arkos\\AppData\\Roaming\\Python\\Python312\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\win32\\lib', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\Pythonwin', 'c:\\ProgramData\\anaconda3\\Lib\\site-packages\\setuptools\\_vendor']


## Module 2: History-Aware State Encoder

This module implements a GRU-based state encoder that processes the full history of patient states up to each timestep, providing a richer representation for downstream tasks.

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass, field
from copy import deepcopy
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors, KernelDensity
from collections import defaultdict, Counter

import warnings
warnings.filterwarnings('ignore')

import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [5]:
# ====================== GLOBAL DATA CONFIG ======================
DATA_DIR = '../../../../../data/'
print(f"✅ Using DATA_DIR: {DATA_DIR}")

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"❌ Data folder not found at {DATA_DIR}\n"
                            f"Please verify the path above.")

# Quick sanity check of required files
required_files = ['ADMISSIONS.csv', 'PATIENTS.csv', 'ICUSTAYS.csv',
                  'DIAGNOSES_ICD.csv', 'CHARTEVENTS.csv', 'LABEVENTS.csv',
                  'OUTPUTEVENTS.csv', 'INPUTEVENTS_MV.csv']
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print(f"⚠️  Missing files: {missing}")
else:
    print("✅ All required CSV files found.")

✅ Using DATA_DIR: ../../../../../data/
✅ All required CSV files found.


In [6]:
# ====================== IMPORT SHARED UTILITIES ======================
# Import shared classes from utils.data_pipeline module
import os
import sys

notebook_dir = os.getcwd()
src_dir = os.path.abspath(os.path.join(notebook_dir, '..'))

if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from utils.data_pipeline import (
    CohortExtractor, FeatureExtractor, FeatureConfig, FeatureImputer,
    fit_imputer_on_cohort, ActionBins, ActionDiscretizer, Trajectory,
    TrajectoryBuilder, DatasetSplitter, FeatureNormalizer
)
print("✅ Successfully imported shared utilities from utils.data_pipeline")


✅ Successfully imported shared utilities from utils.data_pipeline


### 2.1 HISTORY-AWARE STATE ENCODER 

In [7]:
class HistoryAwareStateEncoder(nn.Module):

    def __init__(self,
                 d_state: int = 70,
                 d_hidden: int = 256,
                 d_latent: int = 128,
                 n_layers: int = 2,
                 dropout: float = 0.1):
        super().__init__()
        
        self.d_state = d_state
        self.d_hidden = d_hidden
        self.d_latent = d_latent
        self.n_layers = n_layers
        
        # State embedding (project to hidden dimension)
        self.state_embed = nn.Sequential(
            nn.Linear(d_state, d_hidden),
            nn.LayerNorm(d_hidden),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Recurrent encoder (processes history)
        self.gru = nn.GRU(
            input_size=d_hidden,
            hidden_size=d_hidden,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0,
            batch_first=True
        )
        
        # Projection to latent space
        self.latent_projection = nn.Sequential(
            nn.Linear(d_hidden, d_latent),
            nn.Tanh()
        )
        
        # For pre-training: next-state prediction
        self.dynamics_head = nn.Sequential(
            nn.Linear(d_latent + 25, d_hidden),  # latent + action (one-hot)
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_state)
        )
        
        print(f"✅ Initialized HistoryAwareStateEncoder:")
        print(f"   State dim: {d_state} (38 features + 38 missingness indicators)")
        print(f"   Hidden: {d_hidden}, Latent: {d_latent}")
        print(f"   GRU layers: {n_layers}")
        
    def forward(self, 
                states: torch.Tensor,
                lengths: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Encode state history up to each timestep.
        
        Args:
            states: [B, T, D_state] - state sequences from REAL MIMIC-III data
            lengths: [B] - actual sequence lengths (for padding)
            
        Returns:
            z: [B, T, D_latent] - history-aware encodings at each timestep
        """
        B, T, D = states.shape
        
        # Validation
        assert D == self.d_state, (
            f"Input state dimension {D} does not match model's d_state {self.d_state}. "
            f"Expected 70 features (35 base + 35 missingness from FeatureImputer)."
        )
        
        # Embed states
        x = self.state_embed(states)  # [B, T, D_hidden]
        
        # Pack sequences if lengths provided (handles variable lengths)
        if lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(
                x, 
                lengths.cpu(), 
                batch_first=True, 
                enforce_sorted=False
            )
        
        # Encode with GRU (this is the key: processes full history)
        h, _ = self.gru(x)  # [B, T, D_hidden]
        
        # Unpack if packed
        if lengths is not None:
            h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True)
        
        # Project to latent space
        z = self.latent_projection(h)  # [B, T, D_latent]
        
        return z
    
    def encode_single_timestep(self, state_history: torch.Tensor) -> torch.Tensor:
        """
        Encode up to current timestep (for inference).
        
        Args:
            state_history: [B, T_history, D_state] - REAL feature sequences
        Returns:
            z_current: [B, D_latent]
        """
        z_seq = self.forward(state_history)
        return z_seq[:, -1, :]  # Return last timestep encoding

### 2.2 STATE ENCODER PRE-TRAINER

In [8]:
class StateEncoderPretrainer:

    def __init__(self, 
                 encoder: HistoryAwareStateEncoder,
                 device: str = 'cuda'):
        self.encoder = encoder
        self.device = device
        
        # Contrastive projection head (for CPC loss)
        self.contrastive_head = nn.Sequential(
            nn.Linear(encoder.d_latent, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        ).to(device)
        
        print("✅ Initialized StateEncoderPretrainer")
        print("   Task 1: Next-state prediction")
        print("   Task 2: Contrastive predictive coding")
        
    def pretrain(self,
                 train_loader: torch.utils.data.DataLoader,
                 val_loader: torch.utils.data.DataLoader,
                 n_epochs: int = 30,
                 lr: float = 3e-4,
                 save_path: str = 'state_encoder.pt') -> HistoryAwareStateEncoder:

        optimizer = torch.optim.AdamW(
            list(self.encoder.parameters()) + list(self.contrastive_head.parameters()),
            lr=lr,
            weight_decay=0.01
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=n_epochs
        )
        
        best_val_loss = float('inf')
        
        print(f"\n Pre-training State Encoder for {n_epochs} epochs...")
        print(f"   Using REAL MIMIC-III trajectory data")
        
        for epoch in range(n_epochs):
            # Training
            train_loss, train_metrics = self._train_epoch(train_loader, optimizer)
            
            # Validation
            val_loss, val_metrics = self._validate(val_loader)
            
            scheduler.step()
            
            # Logging
            print(f"\nEpoch {epoch+1}/{n_epochs}")
            print(f"  Train Loss: {train_loss:.4f} | "
                  f"Dynamics: {train_metrics['dynamics']:.4f} | "
                  f"Contrastive: {train_metrics['contrastive']:.4f}")
            print(f"  Val Loss:   {val_loss:.4f} | "
                  f"Dynamics: {val_metrics['dynamics']:.4f} | "
                  f"Contrastive: {val_metrics['contrastive']:.4f}")
            
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    'encoder': self.encoder.state_dict(),
                    'contrastive_head': self.contrastive_head.state_dict(),
                    'epoch': epoch,
                    'val_loss': val_loss
                }, save_path)
                print(f"  ✅ Saved best model (val_loss: {val_loss:.4f})")
        
        # Load best weights
        checkpoint = torch.load(save_path)
        self.encoder.load_state_dict(checkpoint['encoder'])
        self.contrastive_head.load_state_dict(checkpoint['contrastive_head'])
        
        print(f"\n✅ Pre-training complete! Best val loss: {best_val_loss:.4f}")
        
        return self.encoder
    
    def _train_epoch(self, loader, optimizer):
        """Single training epoch - processes REAL trajectories"""
        self.encoder.train()
        self.contrastive_head.train()
        
        total_loss = 0
        total_dynamics = 0
        total_contrastive = 0
        
        for batch in tqdm(loader, desc="Training", leave=False):
            states = batch['states'].to(self.device)  # REAL MIMIC-III features
            actions = batch['actions'].to(self.device)
            lengths = batch['lengths'].to(self.device)
            
            # Encode history (excluding last timestep for next-state prediction)
            z = self.encoder(states[:, :-1], lengths - 1)  # [B, T-1, D_latent]
            
            # Loss 1: Next-state prediction
            actions_onehot = F.one_hot(actions[:, :-1].long(), num_classes=25).float()
            z_with_action = torch.cat([z, actions_onehot], dim=-1)
            
            pred_next_states = self.encoder.dynamics_head(z_with_action)
            actual_next_states = states[:, 1:]
            
            loss_dynamics = F.mse_loss(pred_next_states, actual_next_states)
            
            # Loss 2: Contrastive predictive coding
            if states.shape[1] >= 10:
                z_current = self.contrastive_head(z[:, :-5, :])
                z_future = self.contrastive_head(z[:, 5:, :])
                
                loss_contrastive = self._contrastive_loss(z_current, z_future)
            else:
                loss_contrastive = torch.tensor(0.0, device=self.device)
            
            # Combined loss
            loss = loss_dynamics + 0.5 * loss_contrastive
            
            # Optimize
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.encoder.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            total_dynamics += loss_dynamics.item()
            total_contrastive += loss_contrastive.item()
        
        n_batches = len(loader)
        return total_loss / n_batches, {
            'dynamics': total_dynamics / n_batches,
            'contrastive': total_contrastive / n_batches
        }
    
    def _validate(self, loader):
        """Validation epoch - REAL data only"""
        self.encoder.eval()
        self.contrastive_head.eval()
        
        total_loss = 0
        total_dynamics = 0
        total_contrastive = 0
        
        with torch.no_grad():
            for batch in loader:
                states = batch['states'].to(self.device)
                actions = batch['actions'].to(self.device)
                lengths = batch['lengths'].to(self.device)
                
                z = self.encoder(states[:, :-1], lengths - 1)
                
                actions_onehot = F.one_hot(actions[:, :-1].long(), num_classes=25).float()
                z_with_action = torch.cat([z, actions_onehot], dim=-1)
                
                pred_next_states = self.encoder.dynamics_head(z_with_action)
                loss_dynamics = F.mse_loss(pred_next_states, states[:, 1:])
                
                if states.shape[1] >= 10:
                    z_current = self.contrastive_head(z[:, :-5, :])
                    z_future = self.contrastive_head(z[:, 5:, :])
                    loss_contrastive = self._contrastive_loss(z_current, z_future)
                else:
                    loss_contrastive = torch.tensor(0.0)
                
                loss = loss_dynamics + 0.5 * loss_contrastive
                
                total_loss += loss.item()
                total_dynamics += loss_dynamics.item()
                total_contrastive += loss_contrastive.item()
        
        n_batches = len(loader)
        return total_loss / n_batches, {
            'dynamics': total_dynamics / n_batches,
            'contrastive': total_contrastive / n_batches
        }
    
    def _contrastive_loss(self, z_anchor, z_positive):
        """InfoNCE contrastive loss"""
        B, T, D = z_anchor.shape
        
        z_anchor = z_anchor.reshape(-1, D)
        z_positive = z_positive.reshape(-1, D)
        
        z_anchor = F.normalize(z_anchor, dim=-1)
        z_positive = F.normalize(z_positive, dim=-1)
        
        temperature = 0.1
        sim = torch.mm(z_anchor, z_positive.T) / temperature
        
        labels = torch.arange(z_anchor.shape[0], device=z_anchor.device)
        
        loss = F.cross_entropy(sim, labels)
        
        return loss




### 2.3 DATALOADER UTILITIES

In [9]:
class TrajectoryDataset(torch.utils.data.Dataset):

    def __init__(self, trajectories: List):
        self.trajectories = trajectories
        
        # Validate state dimensions
        if len(trajectories) > 0:
            expected_dim = 76  # 38 features + 38 missingness
            actual_dim = trajectories[0].states.shape[1]
            
            if actual_dim != expected_dim:
                print(f"⚠️  WARNING: Expected state dim {expected_dim}, got {actual_dim}")
                print(f"    Make sure FeatureImputer has added missingness indicators!")
        
    def __len__(self):
        return len(self.trajectories)
    
    def __getitem__(self, idx):
        traj = self.trajectories[idx]
        
        # Convert actions to indices
        if len(traj.actions.shape) == 2:
            # Actions are [T, 2] (fluid_bin, vaso_bin)
            actions = traj.actions[:, 0] * 5 + traj.actions[:, 1]
        else:
            actions = traj.actions
        
        return {
            'states': torch.from_numpy(traj.states).float(),
            'actions': torch.from_numpy(actions).long(),
            'length': traj.length
        }


def collate_trajectories(batch):
    """Collate function for variable-length trajectories"""
    max_length = max(item['length'] for item in batch)
    
    states_padded = []
    actions_padded = []
    lengths = []
    
    for item in batch:
        T = item['length']
        D_state = item['states'].shape[1]
        
        # Pad states
        if T < max_length:
            padding = torch.zeros(max_length - T, D_state)
            states = torch.cat([item['states'], padding], dim=0)
        else:
            states = item['states']
        
        # Pad actions
        if T < max_length:
            actions = torch.cat([
                item['actions'],
                torch.zeros(max_length - T, dtype=torch.long)
            ])
        else:
            actions = item['actions']
        
        states_padded.append(states)
        actions_padded.append(actions)
        lengths.append(T)
    
    return {
        'states': torch.stack(states_padded),
        'actions': torch.stack(actions_padded),
        'lengths': torch.tensor(lengths)
    }


def create_encoder_dataloaders(train_trajectories: List,
                                 val_trajectories: List,
                                 batch_size: int = 32) -> Tuple:

    train_dataset = TrajectoryDataset(train_trajectories)
    val_dataset = TrajectoryDataset(val_trajectories)
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_trajectories,
        num_workers=0
    )
    
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_trajectories,
        num_workers=0
    )
    
    return train_loader, val_loader

In [10]:
def stateEncoder_pretraining_pipeline():
    """
    Complete pipeline for pre-training the history-aware state encoder.
    
    IMPORTANT: This function requires shared classes from module1.ipynb.
    Ensure module1 has been executed in the same kernel before running this.
    """
    
    # Check if shared classes are available
    try:
        CohortExtractor
        FeatureExtractor
        TrajectoryBuilder
    except NameError as e:
        raise NameError(
            f"❌ Missing class: {e}\n"
            "This is because shared classes from module1.ipynb are not available.\n\n"
            "SOLUTION:\n"
            "1. Run module1.ipynb first in the same kernel, OR\n"
            "2. Create utils/data_pipeline.py with all shared classes and import from there\n"
            "See src/README.md for setup instructions."
        )
    
    print("✅ Starting state encoder pre-training pipeline...")
    
    # Step 1: Extract cohort
    print("\n[1/6] Extracting sepsis cohort...")
    extractor = CohortExtractor()
    cohort_df = extractor.extract_cohort()
    
    # Step 2: Initialize components
    print("[2/6] Initializing data processors...")
    feature_extractor = FeatureExtractor(FeatureConfig())
    
    action_discretizer = ActionDiscretizer(
        bins_config=ActionBins(), 
        data_dir='../../../../../data/' 
    )
    
    feature_imputer = fit_imputer_on_cohort(cohort_df, feature_extractor)
    
    # Step 3: Build trajectories
    print("[3/6] Building patient trajectories...")
    traj_builder = TrajectoryBuilder(
        feature_extractor,
        action_discretizer,
        feature_imputer,
        min_length=24,
        max_length=168
    )
    trajectories = traj_builder.build_dataset(cohort_df)
    
    # Step 4: Split
    print("[4/6] Splitting dataset...")
    splitter = DatasetSplitter()
    splits = splitter.split(trajectories)
    
    # Step 5: Normalize
    print("[5/6] Normalizing features...")
    normalizer = FeatureNormalizer()
    normalizer.fit(splits['train'])
    normalized_train = normalizer.transform(splits['train'])
    normalized_val = normalizer.transform(splits['val'])
    
    train_loader, val_loader = create_encoder_dataloaders(
        normalized_train,
        normalized_val,
        batch_size=32
    )
    
    # Step 6: Train encoder
    print("[6/6] Training state encoder...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    encoder = HistoryAwareStateEncoder(
        d_state=76,  
        d_hidden=256,
        d_latent=128,
        n_layers=2,
        dropout=0.1
    ).to(device)
    
    # Pre-train encoder
    pretrainer = StateEncoderPretrainer(encoder, device)
    encoder = pretrainer.pretrain(train_loader, val_loader, n_epochs=30)
    
    print("\n✅ Pipeline complete!")
    return encoder, normalizer


In [11]:
# ====================== IMPORT SHARED UTILITIES ======================
# Import shared classes from utils.data_pipeline module
import sys
import os

# Dynamically resolve the path to the src directory
notebook_dir = os.getcwd()  # Use current working directory for Jupyter
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.insert(0, project_root)

from utils.data_pipeline import (
    CohortExtractor, FeatureExtractor, FeatureConfig, FeatureImputer,
    fit_imputer_on_cohort, ActionBins, ActionDiscretizer, Trajectory,
    TrajectoryBuilder, DatasetSplitter, FeatureNormalizer
)

print("✅ Successfully imported shared utilities from utils.data_pipeline")
print(f"   Available classes: CohortExtractor, FeatureExtractor, FeatureConfig,")
print(f"                      ActionDiscretizer, TrajectoryBuilder, DatasetSplitter,")
print(f"                      FeatureNormalizer")

✅ Successfully imported shared utilities from utils.data_pipeline
   Available classes: CohortExtractor, FeatureExtractor, FeatureConfig,
                      ActionDiscretizer, TrajectoryBuilder, DatasetSplitter,
                      FeatureNormalizer
